# v006 · Train từng model (nhiều hàng song song trên 2 GPU), rồi test một lần + báo cáo

v006 = đội hình v005 train lại trên **seed 43** (`SEEDS`), có **đo dày validation** ở 3 epoch đầu (`settings` → `training.dense_val`: thêm điểm AUC giữa epoch để thấy đường học đi lên từ 0,5; không dùng chọn checkpoint). v005 (seed 42) ở branch `v005`.

Mỗi phiên train (`GROUP` = `P1`…`P6`, `settings.TRAIN_SESSIONS`) là **một notebook Kaggle T4 ×2** tên `vn-av-df-v006-<GROUP>` (viết thường, vd. `vn-av-df-v006-p1`): các hàng chạy song song, **hàng đầu dùng riêng GPU0**, các hàng sau **chia GPU1** (baseline nhẹ chạy đồng thời); các bước trong hàng chạy nối tiếp. Phiên train **chỉ train** (không chấm test). Train xong hết thì chạy `GROUP = "test"`: chấm test mọi model một lần + ghép muộn + cổng + `report.md`, **một run W&B** duy nhất. Đầu mỗi phiên, cache cần được chép sang `/kaggle/temp` bằng 64 luồng (`STAGE_LOCAL`; đọc thẳng `/kaggle/input` chậm vì độ trễ từng file). Bỏ bước nào thì thêm vào `DROP`; chạy một bước lẻ: `GROUP = "<bước>"` (vd. `"M2"`). Thử luồng trước: `SMOKE = True` (vài người mỗi split, 1 epoch, không W&B).

| GROUP | Accelerator | GPU0 | GPU1 |
|---|---|---|---|
| `labels` | None (CPU) | nhãn mismatch 0 / 1 / mask + kiểm lối tắt → `sync_labels.json` (xem trước khi train) | |
| `P1` | T4 ×2 | M3-R (+ benchmark mọi model) | đồng thời: BL-M/sup → M2 → BL-M + SpeechForensics, và BL-A2 → BL-V |
| `P2` (`PREVIOUS_RUN` = P1) | T4 ×2 | M3-R chạy tiếp (xong rồi thì bỏ qua) | đồng thời: M3-R −cặp sạch ↔ nén, và M2 chạy tiếp → BL-M + SpeechForensics |
| `P3` (`PREVIOUS_RUN` = P2, nếu còn model dở) | T4 ×2 | M3-R chạy tiếp | M3-R −cặp chạy tiếp |
| `P4` (`PREVIOUS_RUN` = mọi phiên trước, tuỳ chọn) | T4 ×2 | M3-R +nhánh chất lượng | M3-R không học nén |
| `P5` (`PREVIOUS_RUN` = P1-P4, tuỳ chọn) | T4 ×2 | M3-R +nhánh chất lượng chạy tiếp | M3-R không học nén chạy tiếp → BL-A |
| `P6` (`PREVIOUS_RUN` = P1-P5, tuỳ chọn) | T4 ×2 | BL-A chạy tiếp | M3-R +nhánh chất lượng (nếu còn dở) |
| `test` (`PREVIOUS_RUN` = mọi phiên train) | T4 ×2 | chấm test mọi model (bản gốc + mọi biến thể + PR), ghép muộn BL-Ens / BL-Ens-AV, cổng V / R, `report.md`; một run W&B `test` | |
| `report` | None (CPU) | dựng lại `report.md` từ kết quả test đã có (không chấm lại) | |

Cần: 3 dataset part (có `sync_syncnet.json`, chạy `training/01a_syncnet.py` ở local) + Output prepare: 9 phiên bản gốc + mỗi biến thể 2 phiên × 3 part (`c480`, `c360` bắt buộc; `pr`, `w360`, `c288` tuỳ chọn). Mọi model train thấy các biến thể có `"train": True` (c480, c360; M3-R theo cặp sạch ↔ biến thể, model khác lấy ngẫu nhiên `variant_p` = 0,5; trừ `trihead_v5_clean`, và BL-V / BL-A vì bản biến thể không có khung LIPINC / waveform) và được test thêm trên mọi biến thể (`predictions-test-<biến thể>.npz`, `metrics.json` → `variants`; PR → `pipeline`: AUC khớp đường ống, FPR theo họ đã thấy / chưa thấy). Mỗi run train trên W&B có luôn cách cài phiên (config `setup`, bảng `setup/*`, artifact `setup-log`) và `architecture`; lần test ghi `test-log.json` (số lần model dùng test). Model dừng ở `MAX_HOURS` chưa xong: chạy lại phiên đó với `PREVIOUS_RUN`, train tiếp từ `last.pt`.

In [ ]:
from pathlib import Path
import subprocess, sys, os, time, json, shutil
SESSION_START = time.time()
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-df-forensics.git"  # Sửa thành repository của bạn.
ROOT = Path("/kaggle/working/vn-av-df-forensics")
WORKERS = Path("/kaggle/temp/vn-av-df-workers")  # Môi trường worker: ngoài Output 20 GB.
# Log cài đặt của phiên (git, pip, worker): ghi đủ ra file, cell "Log setup" đưa lên W&B + Output.
LOGS = Path("/kaggle/temp/vn-av-df-logs")
LOGS.mkdir(parents=True, exist_ok=True)
TIMINGS = {}
def logged(name, command, **kw):
    tick = time.time()
    path = LOGS / f"{name}.log"
    with open(path, "a", encoding="utf-8") as log:
        log.write("$ " + " ".join(map(str, command)) + "\n")
        log.flush()
        code = subprocess.run(list(map(str, command)), stdout=log, stderr=subprocess.STDOUT, **kw).returncode
    TIMINGS[name] = round(TIMINGS.get(name, 0) + time.time() - tick, 1)
    if code:
        print("\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]))
        raise RuntimeError(f"{name} lỗi (mã {code}); log đầy đủ: {path}")
if not ROOT.exists():
    logged("git-clone", ["git", "clone", REPO_URL, str(ROOT)])
# Weight và repo upstream (vài GB) ở /kaggle/temp: Output (giới hạn 20 GB) không bao giờ chứa
# chúng, kể cả khi phiên lỗi giữa chừng (bước dọn Output cuối không chạy).
for name in ("weights", "external"):
    (Path("/kaggle/temp/vn-av-df-heavy") / name).mkdir(parents=True, exist_ok=True)
    if not (ROOT / name).exists():
        (ROOT / name).symlink_to(Path("/kaggle/temp/vn-av-df-heavy") / name, target_is_directory=True)
sys.path.insert(0, str(ROOT))
logged("pip-install", [sys.executable, "-m", "pip", "install", "-e", str(ROOT) + "[data,training]"])
import settings
settings.bootstrap()

In [ ]:
# W&B: log quá trình (setup từng phiên, trích feature, nhãn, train, test, báo cáo) làm bằng chứng.
# Project PRIVATE. Thêm secret WANDB_API_KEY: Add-ons → Secrets. Không đẩy ảnh / video người thật.
USE_WANDB = True
WANDB_PROJECT = "vn-av-df-v006"  # v005: "vn-av-df-v005", v004: "vn-av-df-v004" (giữ nguyên, không ghi chồng)
WANDB_ENTITY = "vn-av-df-capstone"  # team trong org vn-av-df-capstone-org; key tạo trong org đó
if USE_WANDB:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_DIR"] = "/kaggle/temp"  # file tạm W&B ở ngoài Output
    os.makedirs(os.environ["WANDB_DIR"], exist_ok=True)
WANDB = {"project": WANDB_PROJECT, "entity": WANDB_ENTITY} if USE_WANDB else None

In [ ]:
GROUP = "labels"      # "labels" → "P1" → "P2" → ("P3") → "test"; tuỳ chọn "P4"-"P6"; hoặc một bước lẻ, vd. "M2"
DROP = []             # bước bỏ ở phiên này, vd. ["M3-abs"] (settings.TRAIN_GROUPS)
DATASET_PARTS = [Path(f"/kaggle/input/datasets/linhxm/{s}") for s in (
    "vn-av-df-data-part1-trim04-fake", "vn-av-df-data-part2-fake", "vn-av-df-data-part3-fake")]
# Output của 9 phiên prepare (3 part × landmarks / cpu / gpu), tên notebook vn-av-df-prepare-p<N>-<phiên>.
PREPARE = "/kaggle/input/notebooks/linhxm/vn-av-df-prepare-p{}-{}/vn-av-df-forensics/cache/vn-av-df-data"
CACHES = [Path(PREPARE.format(p, s)) for p in (1, 2, 3) for s in ("landmarks", "cpu", "gpu")]
# Biến thể (settings.VARIANTS, phiên prepare cùng tên của 3 part): train học biến thể "train" + test thêm trên mọi biến thể.
# Mỗi biến thể 2 phiên mỗi part: "<v>-cpu" (clip nén + mouth) và "<v>" (avhubert + xlsr).
VARIANT_CACHES = {v: [Path(PREPARE.format(p, s)) for p in (1, 2, 3) for s in (v, f"{v}-cpu")] for v in settings.VARIANTS}
# Output các phiên train v006: notebook vn-av-df-v006-<phiên> (v005: vn-av-df-train-<phiên>, runs/train_all_v005).
RUNS = "/kaggle/input/notebooks/linhxm/vn-av-df-v006-{}/vn-av-df-forensics/runs/" + settings.RUN_NAME
PREVIOUS_RUN = []     # P2: [Path(RUNS.format("p1"))]; P3: [Path(RUNS.format("p2"))] (Output P2 đã gộp P1); bước lẻ: Output phiên gần nhất  (slug viết thường)
                      # test / report: Output phiên train cuối (đã gộp các phiên trước), vd. [Path(RUNS.format("p3"))]
SEEDS = list(settings.SEEDS)  # v006: [43] (v005: [42])
MAX_HOURS = 11.5      # tính từ đầu phiên (SESSION_START); dừng sạch (lưu last.pt) trước giới hạn 12 giờ của
                      # Kaggle, chừa ~30 phút cho phần sau train (W&B, Output)
STAGE_LOCAL = True    # chép cache cần sang /kaggle/temp trước khi train / test (nhanh hơn nhiều lần; thiếu chỗ thì tự bỏ)
STAGE_MAX_GB = 90     # trần tổng GB chép: 10/11 phiên GPU ghi ~95 GB rồi lỗi "Read-only file system" dù ổ báo còn chỗ;
                      # bản gốc chép đủ trước, biến thể chia đều phần còn lại, mẫu không chép đọc thẳng /kaggle/input
SMOKE = False         # thử luồng: 2 người mỗi split, 1 epoch, không W&B (xong thì đặt lại False)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # trước lần dùng GPU đầu
import torch
from vn_av_df.dataset import part_selection
for c in CACHES + [c for v, cs in VARIANT_CACHES.items() if settings.VARIANTS[v].get("train") for c in cs]:
    assert c.is_dir(), f"Sai đường dẫn: {c}"
# Biến thể chỉ để đo (PR, w360, c288): thiếu Output phiên prepare thì bỏ, chỉ mất bảng / cổng tương ứng.
for v in [v for v in VARIANT_CACHES if not settings.VARIANTS[v].get("train")]:
    missing = [c for c in VARIANT_CACHES[v] if not c.is_dir()]
    if missing:
        print(f"CẢNH BÁO: thiếu cache biến thể {v} ({len(missing)} part), bỏ biến thể này:", missing[0])
        VARIANT_CACHES.pop(v)
settings.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if GROUP not in ("labels", "report", "test", *settings.TRAIN_SESSIONS, *settings.TRAIN_GROUPS):
    raise ValueError(f"GROUP không có: {GROUP}")
if GROUP not in ("labels", "report") and not torch.cuda.is_available():
    raise RuntimeError(f"GROUP {GROUP} cần Accelerator GPU")
cfg = settings.config()
cfg["deadline"] = SESSION_START + MAX_HOURS * 3600
if SMOKE:
    cfg["smoke_speakers"] = 2
    cfg["training"]["max_epochs"] = 1  # trần trên mọi công thức gốc (BL-M 500, BL-Msup / BL-V / BL-A2 100 epoch)
    WANDB = None
# BL-A trọng số lớp: "balanced" = 1 − tỉ lệ lớp trong train, cùng nguyên tắc với gốc; "paper" =
# đúng 0,1 / 0,9 (cân bằng cho ASVspoof ~90 % giả). Đổi thì BL-A train lại từ đầu.
cfg["recipe_overrides"] = {"ssl_aasist": {"class_weight": "balanced"}}
root, parts = part_selection(DATASET_PARTS)
cfg["dataset"], cfg["dataset_parts"] = str(root), parts
cfg["cache"] = [str(p) for p in CACHES]
cfg["variants"] = {v: [str(p) for p in cs] for v, cs in VARIANT_CACHES.items()}
# Đội hình đầy đủ (settings.TEST_MODELS): để in model nào chưa xong; mỗi bước chỉ train / test model của nó.
cfg["models"], cfg["seeds"] = list(settings.TEST_MODELS), list(SEEDS)
cfg["wandb"] = WANDB
SESSION_NAME = f"train-{GROUP}-{time.strftime('%Y%m%d-%H%M')}"
cfg["session"], cfg["train_group"] = SESSION_NAME, GROUP
from vn_av_df.training.train import merge_runs, missing_runs
from vn_av_df.setup_log import copy_tree
previous = [PREVIOUS_RUN] if isinstance(PREVIOUS_RUN, (str, Path)) else list(PREVIOUS_RUN or [])
for p in previous:
    assert Path(p).is_dir(), f"Sai đường dẫn: {p}"
if previous:
    print("Gộp từ phiên trước:", merge_runs(cfg["runs"], previous))
# Log setup của 9 phiên prepare đi cùng kết quả train (runs/<RUN>/setup/prepare-…).
print("Log setup prepare:", sum((copy_tree(c / "setup", Path(cfg["runs"]) / "setup") for c in CACHES), []))
if GROUP in ("labels", "report", "test"):
    STEPS = []
elif GROUP in settings.TRAIN_SESSIONS:
    STEPS = [[s for s in lane if s not in DROP] for lane in settings.TRAIN_SESSIONS[GROUP]]
else:
    STEPS = [[GROUP]]
# Mỗi hàng một GPU: model của các bước trong hàng, chạy nối tiếp; xong model nào test ngay model đó.
LANES = [[m for s in lane for group in settings.TRAIN_GROUPS[s]["lanes"] for m in group] for lane in STEPS]
LANES = [lane for lane in LANES if lane]
MODELS = [m for lane in LANES for m in lane]  # train ở phiên này
RUN_BENCHMARK = any(settings.TRAIN_GROUPS[s].get("benchmark") for lane in STEPS for s in lane)
cfg["test_after_train"] = False  # v005: test chấm một lần ở GROUP "test"
print("Phiên", GROUP, "| bước:", STEPS, "| model theo GPU:", LANES, "| chưa train xong:", missing_runs(cfg))
# Chép cache model của phiên cần (train + validation, bản gốc + biến thể) sang ổ local bằng 64 luồng: đọc
# /kaggle/input bị chặn bởi độ trễ từng file (đo 08/11: 1 luồng 10 MB/s, 64 luồng 438 MB/s), M3 v004 mất
# 75-96 phút / epoch chỉ vì đọc. Thiếu chỗ thì tự bỏ qua (đọc thẳng như cũ).
if STAGE_LOCAL and (MODELS or GROUP == "test"):
    from vn_av_df.dataset import training_dataset
    from vn_av_df.training.prepare import encoders_for
    from vn_av_df.training.stage import stage_cfg
    tick = time.time()
    _rows, _ = training_dataset(cfg, verify_media=False, labels=False)
    _splits = ("validation", "test") if GROUP == "test" else ("train", "validation")
    print("Ổ /kaggle/temp trước khi chép: còn", round(shutil.disk_usage("/kaggle/temp").free / 1e9, 1), "GB")
    print("Chép sang /kaggle/temp:", stage_cfg(cfg, encoders_for(MODELS or cfg["models"]), _rows, "/kaggle/temp/cache",
                                               splits=_splits, budget_gb=STAGE_MAX_GB))
    print("Ổ /kaggle/temp sau khi chép: còn", round(shutil.disk_usage("/kaggle/temp").free / 1e9, 1), "GB")
    TIMINGS["stage"] = round(time.time() - tick, 1)

In [ ]:
from vn_av_df.cli import model_assets
tick = time.time()
ASSETS = model_assets(cfg) if MODELS or RUN_BENCHMARK else {}  # XLS-R gốc (BL-A) + checkpoint AVH-Align (BL-M, M2, M3)
TIMINGS["assets"] = round(time.time() - tick, 1)
print(ASSETS)

## Log setup

In [ ]:
SETUP_FOLDER = Path(cfg["runs"]) / "setup"
SETUP_INFO = {"group": GROUP, "steps": STEPS, "drop": DROP,
              "titles": {s: settings.TRAIN_GROUPS[s]["title"] for lane in STEPS for s in lane},
              "lanes": LANES, "benchmark": RUN_BENCHMARK, "test_models": cfg["models"], "seeds": SEEDS,
              "previous_run": [str(p) for p in previous], "datasets": [str(p) for p in DATASET_PARTS],
              "caches": [str(p) for p in CACHES], "variant_caches": cfg["variants"], "assets": ASSETS, "max_hours": MAX_HOURS,
              "recipe_overrides": cfg["recipe_overrides"]}

# Ghi cách cài phiên vào Output: commit, phần cứng, gói pip của kernel và từng worker, log cài (pip,
# worker), dữ liệu / cache / tài sản gắn vào. Run W&B của phiên (prepare / train / labels / report) tự
# đính kèm: config `setup`, bảng `setup/*`, artifact `setup-log`. Mở run nào cũng biết nó được cài thế nào.
from vn_av_df.setup_log import log_setup
cfg["setup"] = log_setup(cfg, SESSION_NAME, SETUP_FOLDER, logs=sorted(LOGS.glob("*.log")), env_root=WORKERS,
                         info=SETUP_INFO, timings=TIMINGS)

In [ ]:
if GROUP == "labels":
    from vn_av_df.training.sync_labels import report
    labels = report(cfg)
    print(json.dumps({k: v for k, v in labels.items() if k != "samples"}, indent=1, ensure_ascii=False)[:20000])
    print("→ gửi runs/" + settings.RUN_NAME + "/sync_labels.json để xem trước khi train")
if RUN_BENCHMARK:
    from vn_av_df.training.benchmark import benchmark
    print(json.dumps(benchmark(cfg, cfg["models"], steps=10), indent=2))
if MODELS and len(LANES) > 1 and torch.cuda.is_available():
    from vn_av_df.training.lanes import run_lanes  # hàng đầu GPU0 riêng, hàng sau chia GPU1
    run_lanes(cfg, LANES)
elif MODELS:
    from vn_av_df.training.train import train_all
    print(json.dumps(train_all({**cfg, "models": MODELS}), indent=2, ensure_ascii=False))

In [ ]:
# Phiên train chỉ train; GROUP "test" chấm test mọi model một lần (một run W&B cho test + báo cáo).
from vn_av_df.training.train import run_state
runs = Path(cfg["runs"])
if MODELS:
    done = [m for m in MODELS if all(run_state(runs / f"{m}_seed{s}")[0] for s in SEEDS)]
    print("Train xong:", done, "| còn dở (chạy lại phiên với PREVIOUS_RUN):", [m for m in MODELS if m not in done])
if GROUP == "test":
    from vn_av_df import tracking
    from vn_av_df.training.evaluate import evaluate_all
    from vn_av_df.training.report import build_report
    print("Chưa train xong (không có trong test):", missing_runs(cfg))
    TEST_RUN = tracking.open_run(cfg, "test", f"{runs.name}-test", group=runs.name, folder=runs, with_setup=True)
    print(json.dumps(evaluate_all(cfg, run=TEST_RUN), indent=2))
    report = build_report(cfg, run=TEST_RUN)
    TEST_RUN.finish()
    print(Path(report["report"]).read_text(encoding="utf-8"))
if GROUP == "report":
    from vn_av_df.training.report import build_report
    report = build_report(cfg)  # đọc kết quả test đã có (GROUP "test"), không chấm lại
    print(Path(report["report"]).read_text(encoding="utf-8"))
from vn_av_data.common.runtime import prune_output
print("Đã xoá khỏi Output:", len(prune_output(ROOT, [str(Path(cfg["runs"]).relative_to(ROOT))])), "mục")